# Preprocesamiento de Laboratorio — Datos Tabulares

Extrae resultados de laboratorio de `labevents` para los stays de la cohorte Sepsis-3.

**Salida**: `data/processed/labs/labs_hourly.parquet`  
Formato: una fila por `(stay_id, hours_from_intime)` con columnas por cada lab + indicadores de ausencia.

**Diferencias clave respecto a vitales**:
- `labevents` tiene `hadm_id`, no `stay_id` → join por admisión, asignamos a stay por ventana temporal
- Forward-fill hasta **24h** (los labs persisten más que los vitales)
- Se añade columna `{lab}_measured` (1 si se midió en esa hora, 0 si es carry-forward o nulo)

**Labs extraídos**:

| Variable | itemid(s) | Referencia |
|---|---|---|
| platelet | 51265 | SOFA coagulación |
| bilirubin | 50885 | SOFA hepático |
| creatinine | 50912 | SOFA renal |
| lactate | 50813, 52442 | marcador sepsis |
| wbc | 51301 | SIRS / infección |
| hemoglobin | 51222 | anemia |
| sodium | 50983 | electrolitos |
| potassium | 50971 | electrolitos |
| glucose | 50931 | metabolismo |
| bicarbonate | 50882 | equilibrio ácido-base |
| bun | 51006 | función renal |
| albumin | 50862 | estado nutricional |
| alt | 50861 | función hepática |
| ast | 50878 | función hepática |
| inr | 51237 | coagulación |
| ptt | 51275 | coagulación |
| ph | 50820 | gasometría |
| pco2 | 50818 | gasometría |
| pao2 | 50821 | gasometría / SOFA resp |

In [1]:
import polars as pl
import numpy as np
from pathlib import Path
import time

MIMIC        = Path.home() / "mimic-iv-3.0"
HOSP         = MIMIC / "hosp"
OUT_DIR      = Path("../data/processed/labs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
COHORT_PATH  = Path("../data/processed/cohort.parquet")

def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}")

LAB_ITEMS = {
    "platelet":    [51265],
    "bilirubin":   [50885],
    "creatinine":  [50912],
    "lactate":     [50813, 52442],
    "wbc":         [51301],
    "hemoglobin":  [51222],
    "sodium":      [50983],
    "potassium":   [50971],
    "glucose":     [50931],
    "bicarbonate": [50882],
    "bun":         [51006],
    "albumin":     [50862],
    "alt":         [50861],
    "ast":         [50878],
    "inr":         [51237],
    "ptt":         [51275],
    "ph":          [50820],
    "pco2":        [50818],
    "pao2":        [50821],
}

ALL_LAB_IDS = [iid for ids in LAB_ITEMS.values() for iid in ids]

# Lookup itemid → lab name (evita pl.replace con tipos mixtos)
item_lookup = pl.DataFrame({
    "itemid":   ALL_LAB_IDS,
    "lab_name": [name for name, ids in LAB_ITEMS.items() for _ in ids],
}).with_columns(pl.col("itemid").cast(pl.Int64))

LAB_NAMES = list(LAB_ITEMS.keys())

print(f"Labs a extraer: {len(LAB_ITEMS)}  ({len(ALL_LAB_IDS)} item IDs)")
print(item_lookup)

Labs a extraer: 19  (20 item IDs)
shape: (20, 2)
┌────────┬────────────┐
│ itemid ┆ lab_name   │
│ ---    ┆ ---        │
│ i64    ┆ str        │
╞════════╪════════════╡
│ 51265  ┆ platelet   │
│ 50885  ┆ bilirubin  │
│ 50912  ┆ creatinine │
│ 50813  ┆ lactate    │
│ 52442  ┆ lactate    │
│ …      ┆ …          │
│ 51237  ┆ inr        │
│ 51275  ┆ ptt        │
│ 50820  ┆ ph         │
│ 50818  ┆ pco2       │
│ 50821  ┆ pao2       │
└────────┴────────────┘


## PASO 1 — Cargar cohorte

In [2]:
cohort = pl.read_parquet(COHORT_PATH).select([
    "subject_id", "hadm_id", "stay_id",
    "intime", "outtime", "los_hours",
    "sepsis", "onset_time",
])

hadm_ids = cohort["hadm_id"].unique().to_list()

print(f"Stays:           {len(cohort):,}")
print(f"Admisiones únicas (hadm_id): {len(hadm_ids):,}")
cohort.head(3)

Stays:           74,829
Admisiones únicas (hadm_id): 68,546


[Vista previa de registros individuales omitida — DUA de PhysioNet: los datos de MIMIC-IV no se redistribuyen]


## PASO 2 — Cargar labevents filtrado a la cohorte

`labevents` tiene `hadm_id`, no `stay_id`. Filtramos por `hadm_id` de la cohorte.

In [3]:
log("Leyendo labevents (puede tardar ~90s) ...")

labs_raw = (
    pl.read_csv(
        HOSP / "labevents.csv.gz",
        columns=["hadm_id", "itemid", "charttime", "valuenum"],
    )
    .with_columns([
        pl.col("hadm_id").cast(pl.Int64),
        pl.col("itemid").cast(pl.Int64),
        pl.col("charttime").str.to_datetime(strict=False),
    ])
    .filter(
        pl.col("itemid").is_in(ALL_LAB_IDS) &
        pl.col("hadm_id").is_in(hadm_ids) &
        pl.col("valuenum").is_not_null()
    )
    .join(item_lookup, on="itemid", how="left")
    .drop("itemid")
)

log(f"Registros cargados: {len(labs_raw):,}")
log(f"Admisiones con labs: {labs_raw['hadm_id'].n_unique():,}")
labs_raw.head(5)

[19:39:47] Leyendo labevents (puede tardar ~90s) ...


[19:41:00] Registros cargados: 13,163,086
[19:41:00] Admisiones con labs: 68,328


[Vista previa de registros individuales omitida — DUA de PhysioNet: los datos de MIMIC-IV no se redistribuyen]


In [4]:
# Cobertura por lab
coverage = (
    labs_raw
    .group_by("lab_name")
    .agg([
        pl.col("hadm_id").n_unique().alias("admisiones_con_dato"),
        pl.col("valuenum").count().alias("n_registros"),
        pl.col("valuenum").mean().round(2).alias("media"),
        pl.col("valuenum").std().round(2).alias("std"),
    ])
    .with_columns(
        (pl.col("admisiones_con_dato") / len(hadm_ids) * 100).round(1).alias("cobertura_pct")
    )
    .sort("cobertura_pct", descending=True)
)

print(f"Cobertura por lab (sobre {len(hadm_ids):,} admisiones):")
print(coverage)

Cobertura por lab (sobre 68,546 admisiones):
shape: (19, 6)
┌───────────┬─────────────────────┬─────────────┬────────┬────────┬───────────────┐
│ lab_name  ┆ admisiones_con_dato ┆ n_registros ┆ media  ┆ std    ┆ cobertura_pct │
│ ---       ┆ ---                 ┆ ---         ┆ ---    ┆ ---    ┆ ---           │
│ str       ┆ u32                 ┆ u32         ┆ f64    ┆ f64    ┆ f64           │
╞═══════════╪═════════════════════╪═════════════╪════════╪════════╪═══════════════╡
│ sodium    ┆ 68297               ┆ 1052044     ┆ 138.52 ┆ 5.58   ┆ 99.6          │
│ wbc       ┆ 68266               ┆ 922569      ┆ 11.36  ┆ 15.6   ┆ 99.6          │
│ bun       ┆ 68290               ┆ 1008968     ┆ 30.68  ┆ 24.68  ┆ 99.6          │
│ potassium ┆ 68296               ┆ 1057471     ┆ 4.13   ┆ 0.61   ┆ 99.6          │
│ platelet  ┆ 68269               ┆ 937356      ┆ 216.39 ┆ 140.97 ┆ 99.6          │
│ …         ┆ …                   ┆ …           ┆ …      ┆ …      ┆ …             │
│ pao2      ┆ 49

## PASO 3 — Asignar labs a stays y calcular hora relativa

Un lab pertenece a un stay si su `charttime` cae dentro de `[intime − 24h, outtime]`.  
Las 24h previas al ingreso UCI permiten llevar forward el último lab pre-UCI.

In [5]:
log("Asignando labs a stays por ventana temporal ...")

# Join labs → stays via hadm_id
labs_aligned = (
    labs_raw
    .join(
        cohort.select(["hadm_id", "stay_id", "intime", "outtime", "los_hours"]),
        on="hadm_id",
        how="left",
    )
    .with_columns(
        ((pl.col("charttime") - pl.col("intime")).dt.total_minutes() / 60)
        .floor().cast(pl.Int32).alias("hours_from_intime")
    )
    # Incluir labs hasta 24h antes del ingreso (para carry-forward pre-UCI)
    .filter(
        (pl.col("hours_from_intime") >= -24) &
        (pl.col("hours_from_intime") <= pl.col("los_hours").cast(pl.Int32))
    )
    .select(["stay_id", "hours_from_intime", "lab_name", "valuenum"])
)

log(f"Registros asignados: {len(labs_aligned):,}")
log(f"Stays con labs: {labs_aligned['stay_id'].n_unique():,}")
labs_aligned.head(5)

[19:41:00] Asignando labs a stays por ventana temporal ...


[19:41:01] Registros asignados: 8,561,991
[19:41:01] Stays con labs: 74,532


[Vista previa de registros individuales omitida — DUA de PhysioNet: los datos de MIMIC-IV no se redistribuyen]


## PASO 4 — Agregar por bucket horario y pivotar a formato ancho

In [6]:
log("Agregando por hora y pivotando ...")

# Último valor en cada bucket horario (más relevante que la media para labs)
labs_1h_long = (
    labs_aligned
    .sort(["stay_id", "lab_name", "hours_from_intime"])
    .group_by(["stay_id", "hours_from_intime", "lab_name"])
    .agg(pl.col("valuenum").last().alias("value"))
)

# Marcar horas con medición real (antes de expandir la grilla)
labs_1h_long = labs_1h_long.with_columns(
    pl.lit(1).cast(pl.Int8).alias("measured")
)

# Pivotar a formato ancho
labs_1h = (
    labs_1h_long.drop("measured")
    .pivot(index=["stay_id", "hours_from_intime"], on="lab_name", values="value")
    .sort(["stay_id", "hours_from_intime"])
)

# Tabla de qué labs se midieron en cada hora (para los indicadores de ausencia)
labs_measured = (
    labs_1h_long.drop("value")
    .pivot(index=["stay_id", "hours_from_intime"], on="lab_name", values="measured")
    .sort(["stay_id", "hours_from_intime"])
    .rename({name: f"{name}_measured" for name in LAB_NAMES if name in labs_1h_long["lab_name"].unique().to_list()})
)

# Garantizar columnas para todos los labs
for name in LAB_NAMES:
    if name not in labs_1h.columns:
        labs_1h = labs_1h.with_columns(pl.lit(None, dtype=pl.Float64).alias(name))
    mname = f"{name}_measured"
    if mname not in labs_measured.columns:
        labs_measured = labs_measured.with_columns(pl.lit(None, dtype=pl.Int8).alias(mname))

log(f"Formato ancho: {labs_1h.shape}")

[19:41:01] Agregando por hora y pivotando ...


[19:41:05] Formato ancho: (1162189, 21)


## PASO 5 — Expandir grilla horaria y forward-fill (24h)

Los labs se miden con poca frecuencia (1-2 veces al día).  
Usamos forward-fill hasta **24h** para propagar el último valor conocido.

In [7]:
log("Expandiendo grilla horaria (-24 a los_hours) ...")

# Grilla desde -24h (pre-UCI) hasta fin de estancia
stay_ranges = cohort.select(["stay_id", "los_hours"]).with_columns(
    pl.col("los_hours").cast(pl.Int32)
)

hour_grid = (
    stay_ranges
    .with_columns(
        pl.int_ranges(-24, pl.col("los_hours") + 1).alias("hours_from_intime")
    )
    .explode("hours_from_intime")
    .select(["stay_id", "hours_from_intime"])
)

log(f"Grilla completa: {len(hour_grid):,} filas")

# Join con datos reales
labs_full = (
    hour_grid
    .join(labs_1h,      on=["stay_id", "hours_from_intime"], how="left")
    .join(labs_measured, on=["stay_id", "hours_from_intime"], how="left")
    .sort(["stay_id", "hours_from_intime"])
)

log(f"Grilla con datos: {labs_full.shape}")

[19:41:05] Expandiendo grilla horaria (-24 a los_hours) ...
[19:41:05] Grilla completa: 9,736,132 filas


[19:41:06] Grilla con datos: (9736132, 40)


In [8]:
log("Aplicando forward-fill (máx 24h) ...")

# Forward-fill valores (24h)
ffill_exprs = [
    pl.col(name).forward_fill(limit=24).over("stay_id")
    for name in LAB_NAMES
    if name in labs_full.columns
]

# Los indicadores _measured: 1 si medido en ESA hora, 0 si carry-forward o null
measured_exprs = [
    pl.col(f"{name}_measured").fill_null(0).cast(pl.Int8)
    for name in LAB_NAMES
    if f"{name}_measured" in labs_full.columns
]

labs_ffill = labs_full.with_columns(ffill_exprs + measured_exprs)

# Nulos tras ffill (labs que nunca se midieron en la estancia)
print("% nulos tras forward-fill (24h):")
null_stats = [
    (name, round(100 * labs_ffill[name].is_null().sum() / len(labs_ffill), 1))
    for name in LAB_NAMES
    if name in labs_ffill.columns
]
for name, pct in sorted(null_stats, key=lambda x: x[1]):
    bar = '█' * int(pct / 4)
    print(f"  {name:<14} {pct:>5}%  {bar}")

[19:41:06] Aplicando forward-fill (máx 24h) ...


% nulos tras forward-fill (24h):
  creatinine      19.5%  ████
  bun             19.5%  ████
  bicarbonate     19.6%  ████
  potassium       19.7%  ████
  sodium          19.8%  ████
  platelet        20.2%  █████
  wbc             20.2%  █████
  hemoglobin      20.2%  █████
  glucose         20.5%  █████
  inr             40.4%  ██████████
  ptt             40.8%  ██████████
  ph              55.5%  █████████████
  pco2            58.0%  ██████████████
  pao2            58.0%  ██████████████
  lactate         65.7%  ████████████████
  ast             67.2%  ████████████████
  alt             67.6%  ████████████████
  bilirubin       67.8%  ████████████████
  albumin         82.1%  ████████████████████


## PASO 6 — Recortar a horas ≥ 0 y guardar

Las horas negativas (−24 a −1) solo se usaron para el carry-forward pre-UCI.  
El parquet final empieza en `hours_from_intime = 0` (momento del ingreso UCI).

In [9]:
log("Recortando a hours_from_intime >= 0 ...")

# Columnas finales: stay_id, hours, valores, indicadores de medición
val_cols  = [name for name in LAB_NAMES if name in labs_ffill.columns]
meas_cols = [f"{name}_measured" for name in LAB_NAMES if f"{name}_measured" in labs_ffill.columns]

labs_final = (
    labs_ffill
    .filter(pl.col("hours_from_intime") >= 0)
    .select(["stay_id", "hours_from_intime"] + val_cols + meas_cols)
)

# Añadir etiquetas
cohort_labels = cohort.select(["stay_id", "intime", "sepsis", "onset_time"]).with_columns(
    pl.when(pl.col("onset_time").is_not_null())
    .then(
        ((pl.col("onset_time") - pl.col("intime")).dt.total_minutes() / 60)
        .floor().cast(pl.Int32)
    )
    .otherwise(None)
    .alias("onset_hour")
)

labs_labeled = labs_final.join(
    cohort_labels.select(["stay_id", "sepsis", "onset_hour"]),
    on="stay_id",
    how="left",
)

output_path = OUT_DIR / "labs_hourly.parquet"
labs_labeled.write_parquet(output_path)
log(f"Guardado en: {output_path}")

total_stays  = labs_labeled["stay_id"].n_unique()
sepsis_stays = labs_labeled.filter(pl.col("sepsis") == 1)["stay_id"].n_unique()
no_sep_stays = labs_labeled.filter(pl.col("sepsis") == 0)["stay_id"].n_unique()

print("=" * 50)
print("  LABS — COMPLETADO")
print("=" * 50)
print(f"  Stays totales:          {total_stays:,}")
print(f"  Stays con sepsis:       {sepsis_stays:,}")
print(f"  Stays sin sepsis:       {no_sep_stays:,}")
print(f"  Filas totales:          {len(labs_labeled):,}")
print(f"  Media horas/stay:       {len(labs_labeled)/total_stays:.1f}")
print(f"  Columnas:               {len(labs_labeled.columns)}")
print(f"  Archivo: {output_path}")
print(f"\nColumnas: {labs_labeled.columns}")

[19:41:06] Recortando a hours_from_intime >= 0 ...


[19:41:07] Guardado en: ../data/processed/labs/labs_hourly.parquet


  LABS — COMPLETADO
  Stays totales:          74,829
  Stays con sepsis:       17,189
  Stays sin sepsis:       57,640
  Filas totales:          7,940,236
  Media horas/stay:       106.1
  Columnas:               42
  Archivo: ../data/processed/labs/labs_hourly.parquet

Columnas: ['stay_id', 'hours_from_intime', 'platelet', 'bilirubin', 'creatinine', 'lactate', 'wbc', 'hemoglobin', 'sodium', 'potassium', 'glucose', 'bicarbonate', 'bun', 'albumin', 'alt', 'ast', 'inr', 'ptt', 'ph', 'pco2', 'pao2', 'platelet_measured', 'bilirubin_measured', 'creatinine_measured', 'lactate_measured', 'wbc_measured', 'hemoglobin_measured', 'sodium_measured', 'potassium_measured', 'glucose_measured', 'bicarbonate_measured', 'bun_measured', 'albumin_measured', 'alt_measured', 'ast_measured', 'inr_measured', 'ptt_measured', 'ph_measured', 'pco2_measured', 'pao2_measured', 'sepsis', 'onset_hour']
